In [1]:
#1B. Cellule 1 : outils

import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
import ee
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_tools
ee.Initialize(project='tefesat')
print('OK')

OK


In [2]:
#1C. Cellule 2 : zone sud

polygon = [[[-16.54, 15.87], [-16.47, 15.87], [-16.47, 15.98],
            [-16.54, 15.98], [-16.54, 15.87]]]
polygon = SDS_tools.smallest_rectangle(polygon)

inputs = {
    'polygon': polygon,
    'dates': ['2015-07-01', '2025-12-31'],
    'sat_list': ['S2'],
    'sitename': 'SAINT_LOUIS_SUD',
    'filepath': os.path.join(os.getcwd(), 'data'),
    'landsat_collection': 'C02'
}
print('Zone sud définie')

Zone sud définie


In [3]:
#1D. Cellule 3 : téléchargement
result = SDS_download.check_images_available(inputs)
metadata = SDS_download.retrieve_images(inputs)
metadata = SDS_download.get_metadata(inputs)

Number of images available between 2015-07-01 and 2025-12-31:
- In Landsat Tier 1 & Sentinel-2 Level-1C:
     S2: 678 images
  Total to download: 678 images
S2: 678/678 images already exist, 0 to be downloaded
Number of images available between 2015-07-01 and 2025-12-31:
- In Landsat Tier 1 & Sentinel-2 Level-1C:
     S2: 678 images
  Total to download: 678 images
S2: 678/678 images already exist, 0 to be downloaded

S2: 0 images

Satellite images downloaded from GEE and save in C:\MonCarriere\tefeSAT\Hackaton_Sensat\CoastSat\data\SAINT_LOUIS_SUD


In [7]:
#Cellule 4 : réglages et ligne de référence

settings = {
    'cloud_thresh': 0.1, 'dist_clouds': 300, 'output_epsg': 32628,
    'check_detection': False, 'adjust_detection': False, 'save_figure': False,
    'min_beach_area': 1000, 'min_length_sl': 500, 'cloud_mask_issue': False,
    'sand_color': 'default', 'pan_off': False, 's2cloudless_prob': 40,
    'inputs': inputs
}
%matplotlib qt
settings['reference_shoreline'] = SDS_preprocess.get_reference_sl(metadata, settings)
settings['max_dist_ref'] = 100

Reference shoreline has been saved in C:\MonCarriere\tefeSAT\Hackaton_Sensat\CoastSat\data\SAINT_LOUIS_SUD


In [8]:
#Cellule 5 : extraction (la longue, 10 à 30 minutes)

%matplotlib inline
output = SDS_shoreline.extract_shorelines(metadata, settings)
output = SDS_tools.remove_duplicates(output)
output = SDS_tools.remove_inaccurate_georef(output, 10)
with open(os.path.join(inputs['filepath'], inputs['sitename'],ç 'output_sauvegarde.pkl'), 'wb') as f:
    pickle.dump(output, f)
print('Traits de côte extraits :', len(output['dates']))

Mapping shorelines:
S2:   100%
19 duplicates
4 bad georef
Traits de côte extraits : 427


In [9]:
#Cellule 6 (notebook sud) : nettoyer et exporter

import geopandas as gpd
from shapely.geometry import LineString

def garder_ligne_principale(sl, saut=30):
    if sl.size == 0:
        return None
    sl = sl[~np.isnan(sl).any(axis=1)]
    if len(sl) < 2:
        return None
    pas = np.hypot(sl[1:, 0] - sl[:-1, 0], sl[1:, 1] - sl[:-1, 1])
    morceaux = np.split(sl, np.where(pas > saut)[0] + 1)
    cand = []
    for m in morceaux:
        if len(m) < 2:
            continue
        if np.hypot(m[0, 0] - m[-1, 0], m[0, 1] - m[-1, 1]) < saut:
            continue
        cand.append((LineString(m).length, m))
    return max(cand, key=lambda c: c[0])[1] if cand else None

rows = []
for i, d in enumerate(output['dates']):
    p = garder_ligne_principale(np.array(output['shorelines'][i]))
    if p is None:
        continue
    l = LineString(p)
    rows.append({'date': d.strftime('%Y-%m-%d'), 'satellite': output['satname'][i],
                 'precision_m': output['geoaccuracy'][i], 'longueur_m': l.length, 'geometry': l})

gdf = gpd.GeoDataFrame(rows, crs='EPSG:32628')
gdf = gdf.sort_values('longueur_m').drop_duplicates('date', keep='last').sort_values('date')
med = gdf['longueur_m'].median()
bon = gdf[gdf['longueur_m'].between(0.6 * med, 1.4 * med)].copy()

def rugosite(g):
    c = np.array(g.coords)
    d = np.hypot(c[-1, 0] - c[0, 0], c[-1, 1] - c[0, 1])
    return g.length / d if d > 0 else np.inf

bon['rug'] = bon.geometry.apply(rugosite)
m = bon['rug'].median()
mad = (bon['rug'] - m).abs().median() * 1.4826
bon = bon[bon['rug'] <= m + 3 * mad].copy()
bon['annee'] = bon['date'].str[:4].astype(int)

print('Traits de côte sud gardés :', len(bon), 'sur', len(gdf))
print(bon.groupby('annee').size().to_string())

final = bon[['date', 'satellite', 'precision_m', 'geometry']].to_crs(epsg=4326)
os.makedirs(os.path.join('..', 'donnees'), exist_ok=True)
final.to_file(os.path.join('..', 'donnees', 'traits_de_cote_sud.geojson'), driver='GeoJSON')
print('Exporté : donnees/traits_de_cote_sud.geojson')

Traits de côte sud gardés : 220 sur 405
annee
2015     1
2016    14
2017    22
2018    23
2019    26
2020    22
2021    18
2022    23
2023    25
2024    21
2025    25
Exporté : donnees/traits_de_cote_sud.geojson


In [10]:
#Fusion nord et sud : un seul fichier de côte

import geopandas as gpd
from shapely.geometry import LineString
from pyproj import Transformer

nord = gpd.read_file(os.path.join('..', 'donnees', 'traits_de_cote.geojson')).to_crs(32628).set_index('date')
sud = gpd.read_file(os.path.join('..', 'donnees', 'traits_de_cote_sud.geojson')).to_crs(32628).set_index('date')
y_cut = Transformer.from_crs(4326, 32628, always_xy=True).transform(-16.51, 15.975)[1]

def morceau(geom, sud_ou_nord):
    c = np.array(geom.coords)
    c = c[np.argsort(c[:, 1])]
    return c[c[:, 1] < y_cut] if sud_ou_nord == 'sud' else c[c[:, 1] >= y_cut]

rows = []
for d in sorted(set(nord.index) & set(sud.index)):
    cs, cn = morceau(sud.loc[d, 'geometry'], 'sud'), morceau(nord.loc[d, 'geometry'], 'nord')
    if len(cs) < 2 or len(cn) < 2:
        continue
    rows.append({'date': d, 'satellite': nord.loc[d, 'satellite'],
                 'precision_m': max(nord.loc[d, 'precision_m'], sud.loc[d, 'precision_m']),
                 'geometry': LineString(np.vstack([cs, cn]))})

complet = gpd.GeoDataFrame(rows, crs='EPSG:32628').to_crs(epsg=4326)
complet.to_file(os.path.join('..', 'donnees', 'traits_de_cote_complet.geojson'), driver='GeoJSON')
print('Dates communes :', len(complet))
print(complet.groupby(complet['date'].str[:4]).size().to_string())
print('Emprise :', complet.total_bounds.round(3))

Dates communes : 199


AttributeError: Can only use .str accessor with string values, not datetime64

In [11]:
import pandas as pd, json, re

complet['date'] = pd.to_datetime(complet['date']).dt.strftime('%Y-%m-%d')
chemin = os.path.join('..', 'donnees', 'traits_de_cote_complet.geojson')
complet.to_file(chemin, driver='GeoJSON')

dates = [f['properties']['date'] for f in json.load(open(chemin, encoding='utf-8'))['features']]
print('Lignes :', len(dates), '| uniques :', len(set(dates)))
print('Format AAAA-MM-JJ :', all(re.match(r'^\d{4}-\d{2}-\d{2}$', x) for x in dates))
print(complet.groupby(complet['date'].str[:4]).size().to_string())
print('Emprise :', complet.total_bounds.round(3))

Lignes : 199 | uniques : 199
Format AAAA-MM-JJ : True
date
2015     1
2016    11
2017    21
2018    20
2019    23
2020    20
2021    17
2022    20
2023    24
2024    20
2025    22
Emprise : [-16.525  15.87  -16.509  16.06 ]
